# Deep Learning Practical Exam – Set A

## Setup

In [ ]:
# Install these once if they are not already installed
# %pip install tensorflow opencv-python transformers pandas numpy matplotlib scikit-learn

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, mean_squared_error

os.makedirs("outputs", exist_ok=True)

### Task 1.1 — Load and prepare the data

In [ ]:
# Load the dataset
url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
try:
    diabetes = pd.read_csv("diabetes.csv")
except FileNotFoundError:
    diabetes = pd.read_csv(url)

print("Shape:", diabetes.shape)
display(diabetes.head(5))
print("\nMissing values:")
print(diabetes.isnull().sum())

# In these columns, zero is treated as missing
zero_columns = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in zero_columns:
    diabetes[col] = diabetes[col].replace(0, np.nan)
    diabetes[col] = diabetes[col].fillna(diabetes[col].median())

X = diabetes.drop("Outcome", axis=1)
y = diabetes["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

### Task 1.2 — Build the ANN

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

ann = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(16, activation="relu"),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
ann.summary()

### Task 1.3 — Train and evaluate

In [ ]:
start = time.time()
ann_history = ann.fit(
    X_train, y_train,
    epochs=50, batch_size=32, validation_split=0.2, verbose=1
)
ann_minutes = (time.time() - start) / 60

plt.figure(figsize=(7, 4))
plt.plot(ann_history.history["accuracy"], label="Training accuracy")
plt.plot(ann_history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("ANN accuracy")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(ann_history.history["loss"], label="Training loss")
plt.plot(ann_history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("ANN loss")
plt.show()

ann_prob = ann.predict(X_test, verbose=0).ravel()
ann_pred = (ann_prob >= 0.5).astype(int)

ann_accuracy = accuracy_score(y_test, ann_pred)
ann_precision = precision_score(y_test, ann_pred, zero_division=0)
ann_recall = recall_score(y_test, ann_pred, zero_division=0)
ann_f1 = f1_score(y_test, ann_pred, zero_division=0)

print("Test Accuracy:", round(ann_accuracy, 4))
print("Precision:", round(ann_precision, 4))
print("Recall:", round(ann_recall, 4))
print("F1-Score:", round(ann_f1, 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, ann_pred))

### Task 1.4 — Save and reflect

In [ ]:
ann.save("ann_diabetes_model.h5")
print("Saved ann_diabetes_model.h5")

### Task 2.1 — Load and display five images

In [ ]:
import cv2
from tensorflow.keras.datasets import cifar10

# Use five sample images and save them as files
(sample_images, sample_labels), _ = cifar10.load_data()
image_paths = []

for i in range(5):
    path = f"outputs/opencv_original_{i+1}.jpg"
    rgb_image = sample_images[i]
    cv2.imwrite(path, cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR))
    image_paths.append(path)

original_images = []
for path in image_paths:
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Could not read {path}")
    original_images.append(img)
    print(path, "shape (height, width, channels):", img.shape)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, (ax, img) in enumerate(zip(axes, original_images)):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Image {i+1}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Task 2.2 — Apply preprocessing operations

In [ ]:
processed_images = []

for i, img in enumerate(original_images):
    resized = cv2.resize(img, (128, 128))
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 100, 200)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contour_img = resized.copy()
    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y0, w, h = cv2.boundingRect(largest)
        cv2.rectangle(contour_img, (x, y0), (x+w, y0+h), (0, 255, 0), 2)

    cv2.imwrite(f"outputs/opencv_resized_{i+1}.jpg", resized)
    cv2.imwrite(f"outputs/opencv_gray_{i+1}.jpg", gray)
    cv2.imwrite(f"outputs/opencv_edges_{i+1}.jpg", edges)
    cv2.imwrite(f"outputs/opencv_contour_{i+1}.jpg", contour_img)
    processed_images.append((resized, gray, edges, contour_img))

### Task 2.3 — Show the 5 by 5 grid

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(12, 14))
for i, img in enumerate(original_images):
    resized, gray, edges, contour_img = processed_images[i]
    images = [
        cv2.cvtColor(img, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(resized, cv2.COLOR_BGR2RGB),
        gray,
        edges,
        cv2.cvtColor(contour_img, cv2.COLOR_BGR2RGB)
    ]
    titles = ["Original", "Resized", "Grayscale", "Edges", "Contour"]
    for j in range(5):
        axes[i, j].imshow(images[j], cmap="gray" if j in [2, 3] else None)
        axes[i, j].set_title(titles[j] if i == 0 else "")
        axes[i, j].axis("off")
plt.tight_layout()
plt.savefig("outputs/opencv_preprocessing_grid.png")
plt.show()

## Step 3 — CNN: CIFAR-10 image classification

### Task 3.1 — Load and prepare the data

In [ ]:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

class_names = ["airplane", "car", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

(X_train_cnn, y_train_cnn), (X_test_cnn, y_test_cnn) = cifar10.load_data()
X_train_cnn = X_train_cnn[:20000].astype("float32") / 255.0
y_train_cnn = y_train_cnn[:20000]
X_test_cnn = X_test_cnn[:4000].astype("float32") / 255.0
y_test_cnn = y_test_cnn[:4000]

y_train_cat = to_categorical(y_train_cnn, 10)
y_test_cat = to_categorical(y_test_cnn, 10)

fig, axes = plt.subplots(1, 10, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_train_cnn[i])
    ax.set_title(class_names[int(y_train_cnn[i][0])], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Training data:", X_train_cnn.shape)
print("Testing data:", X_test_cnn.shape)

### Task 3.2 — Build the CNN

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dropout

cnn = Sequential([
    Input(shape=(32, 32, 3)),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")
])

cnn.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
cnn.summary()

### Task 3.3 — Train and evaluate

In [ ]:
start = time.time()
cnn_history = cnn.fit(
    X_train_cnn, y_train_cat,
    epochs=10, batch_size=64, validation_split=0.1, verbose=1
)
cnn_minutes = (time.time() - start) / 60

plt.figure(figsize=(7, 4))
plt.plot(cnn_history.history["accuracy"], label="Training accuracy")
plt.plot(cnn_history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("CNN accuracy")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(cnn_history.history["loss"], label="Training loss")
plt.plot(cnn_history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("CNN loss")
plt.show()

cnn_test_loss, cnn_test_accuracy = cnn.evaluate(X_test_cnn, y_test_cat, verbose=0)
cnn_prob = cnn.predict(X_test_cnn, verbose=0)
cnn_pred = np.argmax(cnn_prob, axis=1)
cnn_true = y_test_cnn.ravel()

print("Test Accuracy:", round(cnn_test_accuracy, 4))
print(classification_report(cnn_true, cnn_pred, target_names=class_names, zero_division=0))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test_cnn[i])
    true_name = class_names[int(cnn_true[i])]
    pred_name = class_names[int(cnn_pred[i])]
    ax.set_title(f"True: {true_name}\nPred: {pred_name}",
                 color="red" if true_name != pred_name else "black", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Task 3.4 — Save and reflect

In [ ]:
cnn.save("cnn_cifar10_model.h5")
print("Saved cnn_cifar10_model.h5")

## Step 4 — RNN: Monthly airline passenger prediction

### Task 4.1 — Load and prepare the time series

In [ ]:
airline_url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
airline = pd.read_csv(airline_url)
print(airline.head(5))

plt.figure(figsize=(10, 4))
plt.plot(airline["Passengers"])
plt.title("Monthly airline passengers")
plt.xlabel("Month index")
plt.ylabel("Passengers")
plt.show()

passengers = airline[["Passengers"]].values.astype("float32")
rnn_scaler = MinMaxScaler()
passengers_scaled = rnn_scaler.fit_transform(passengers)

look_back = 12
X_seq, y_seq = [], []
for i in range(look_back, len(passengers_scaled)):
    X_seq.append(passengers_scaled[i-look_back:i, 0])
    y_seq.append(passengers_scaled[i, 0])

X_seq = np.array(X_seq).reshape(-1, look_back, 1)
y_seq = np.array(y_seq)

split_at = int(len(X_seq) * 0.8)
X_train_rnn, X_test_rnn = X_seq[:split_at], X_seq[split_at:]
y_train_rnn, y_test_rnn = y_seq[:split_at], y_seq[split_at:]

print("Training shape:", X_train_rnn.shape)
print("Testing shape:", X_test_rnn.shape)

### Task 4.2 — Build the RNN

In [ ]:
from tensorflow.keras.layers import SimpleRNN

rnn = Sequential([
    Input(shape=(12, 1)),
    SimpleRNN(50, activation="tanh"),
    Dense(1)
])
rnn.compile(optimizer="adam", loss="mse")
rnn.summary()

### Task 4.3 — Train and evaluate

In [ ]:
start = time.time()
rnn_history = rnn.fit(X_train_rnn, y_train_rnn, epochs=50, batch_size=16, verbose=1)
rnn_minutes = (time.time() - start) / 60

plt.figure(figsize=(7, 4))
plt.plot(rnn_history.history["loss"])
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("RNN training loss")
plt.show()

train_pred_scaled = rnn.predict(X_train_rnn, verbose=0)
test_pred_scaled = rnn.predict(X_test_rnn, verbose=0)

train_pred = rnn_scaler.inverse_transform(train_pred_scaled)
test_pred = rnn_scaler.inverse_transform(test_pred_scaled)
train_actual = rnn_scaler.inverse_transform(y_train_rnn.reshape(-1, 1))
test_actual = rnn_scaler.inverse_transform(y_test_rnn.reshape(-1, 1))

rnn_rmse = np.sqrt(mean_squared_error(test_actual, test_pred))
print("Test RMSE:", round(rnn_rmse, 2))

plt.figure(figsize=(10, 4))
plt.plot(range(len(train_actual)), train_actual, label="Actual train")
plt.plot(range(len(train_pred)), train_pred, label="Predicted train")
test_start = len(train_actual)
plt.plot(range(test_start, test_start + len(test_actual)), test_actual, label="Actual test")
plt.plot(range(test_start, test_start + len(test_pred)), test_pred, label="Predicted test")
plt.xlabel("Sequence index")
plt.ylabel("Passengers")
plt.title("Actual vs predicted passengers")
plt.legend()
plt.show()

### Task 4.4 — Save and reflect

In [ ]:
rnn.save("rnn_airline_model.h5")
print("Saved rnn_airline_model.h5")

## Step 5 — Transformer: Movie review sentiment

### Task 5.1 — Set up the pipeline and write 25 sentences

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentences = [
    "The film was absolutely brilliant and moving.",
    "I loved the story and the performances were excellent.",
    "A beautiful movie with a satisfying ending.",
    "The actors made every scene enjoyable.",
    "This is one of the best films I have watched.",
    "The music and visuals were wonderful.",
    "The movie kept me interested from start to finish.",
    "A funny and heartwarming film.",
    "The director did a fantastic job.",
    "I would happily watch this movie again.",
    "Worst movie I have ever seen.",
    "The story was boring and confusing.",
    "The acting was terrible.",
    "I regret spending time on this film.",
    "Nothing about the movie worked for me.",
    "The ending was disappointing.",
    "The jokes were not funny at all.",
    "The film felt slow and dull.",
    "The characters were poorly written.",
    "I would not recommend this movie.",
    "The acting was decent but the plot was weak.",
    "The movie was timepass only.",
    "The film had some good parts and some bad parts.",
    "The story was average.",
    "Total bakwaas picture, but the songs were okay."
]

true_sentiment = ["POS"] * 10 + ["NEG"] * 10 + ["NEU"] * 5
print("Number of sentences:", len(sentences))

### Task 5.2 — Run inference and check the results

In [ ]:
transformer_results = classifier(sentences, truncation=True)
rows = []

for sentence, true_label, result in zip(sentences, true_sentiment, transformer_results):
    rows.append({
        "Sentence": sentence,
        "True_Sentiment": true_label,
        "Predicted_Label": result["label"],
        "Confidence_Score": result["score"]
    })

sentiment_df = pd.DataFrame(rows)
display(sentiment_df)

# Check the 20 clearly positive/negative examples
clear_df = sentiment_df.iloc[:20]
correct_count = sum(
    (row.True_Sentiment == "POS" and row.Predicted_Label == "POSITIVE") or
    (row.True_Sentiment == "NEG" and row.Predicted_Label == "NEGATIVE")
    for row in clear_df.itertuples()
)
print("Correct predictions out of 20 clear examples:", correct_count)

print("\nFive ambiguous / neutral sentences:")
display(sentiment_df.iloc[20:])

### Task 5.3 — Plot confidence and reflect

In [ ]:
bar_colors = ["green" if label == "POSITIVE" else "red"
              for label in sentiment_df["Predicted_Label"]]

plt.figure(figsize=(10, 7))
plt.barh(np.arange(1, 26), sentiment_df["Confidence_Score"], color=bar_colors)
plt.yticks(np.arange(1, 26))
plt.xlabel("Confidence score")
plt.ylabel("Sentence index")
plt.title("Transformer sentiment confidence")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Step 6 — Results, reflection and submission

### Task 6.1 — Results summary

In [ ]:
summary = pd.DataFrame([
    {"Topic": "ANN", "Task type": "Classification", "Dataset": f"Diabetes ({len(diabetes)} rows)",
     "Key metric": "Accuracy", "Result": round(ann_accuracy, 4), "Time (minutes)": round(ann_minutes, 1)},
    {"Topic": "OpenCV", "Task type": "Preprocessing", "Dataset": "5 CIFAR-10 sample images",
     "Key metric": "N/A", "Result": "5 images processed", "Time (minutes)": "Fill in"},
    {"Topic": "CNN", "Task type": "Classification", "Dataset": "CIFAR-10 (20,000 train / 4,000 test)",
     "Key metric": "Accuracy", "Result": round(float(cnn_test_accuracy), 4), "Time (minutes)": round(cnn_minutes, 1)},
    {"Topic": "RNN", "Task type": "Regression / forecasting", "Dataset": f"Airline passengers ({len(airline)} months)",
     "Key metric": "RMSE", "Result": round(float(rnn_rmse), 2), "Time (minutes)": round(rnn_minutes, 1)},
    {"Topic": "Transformer", "Task type": "Inference", "Dataset": "25 custom movie review sentences",
     "Key metric": "Correct of 20 clear examples", "Result": f"{correct_count}/20", "Time (minutes)": "Fill in"}
])
display(summary)

### Task 6.2 — Written reflection

### Task 6.3 — GitHub submission checklist

### Task 6.4 — Summary report starter